## RAG - Data Ingestion Pipeline

### 1. Data Ingestion or Importing data

Let's utilise two different Loaders to load Text and PDF files in our Data folder

In [1]:
all_documents = []

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader


text_loader = DirectoryLoader(
    "Data/Text_Files",
    glob = "*.txt",
    loader_cls = TextLoader,
    loader_kwargs = {"encoding" : "utf-8"},
    show_progress = True
)

text_docs = text_loader.load()

C:\Users\NAMAN KUMAR\AppData\Local\Temp\ipykernel_6244\3003747958.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader
d:\Namit Kumar\Learning\Agentic AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 2/2 [00:00<00:00, 23.99it/s]


In [3]:
all_documents.extend(text_docs) # extend function extends a list but append will add the whole list as one item so it will be a list of list, not list of Document, so use extend

In [4]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader


pdf_loader = DirectoryLoader(
    "Data/PDF_Files",
    glob = "*.pdf",
    loader_cls = PyPDFLoader,
    show_progress = True
)

pdf_docs = pdf_loader.load()

100%|██████████| 2/2 [00:09<00:00,  4.77s/it]


In [5]:
all_documents.extend(pdf_docs)

In [6]:
all_documents

[Document(metadata={'source': 'Data\\Text_Files\\machine_learning.txt'}, page_content="Machine Learning is a branch of artificial intelligence where computers learn from data and identify patterns without being explicitly programmed. Instead of following hard-coded rules, algorithms analyze vast datasets, adjust their internal parameters, and continuously improve their accuracy over time to make automated predictions or decisions.\n\n\nCore MethodologiesMachine learning is broadly categorized into three primary learning methods, each suited for different types of problems and data:Supervised Learning: The algorithm is trained on labeled data, meaning the training set already includes the correct answers. This is ideal for classification (e.g., sorting emails into spam or primary) and regression (e.g., predicting real estate prices based on historical features).Unsupervised Learning: The system analyzes raw, unlabeled data to discover hidden structures, groupings, or patterns. Retailers

In [7]:
len(all_documents)

254

### 2. Chunking

For this we will use RecursiveCharacterTextSplitter from langchain_text_splitter because its is one of the industry Baseline and it is fast and simple. Although there are Sematic text splitters which works on embeddings but they are slow and if we are using a Semantic Document Loader (UnstructuredLoader with mode = "elements") its not bad to use RecursiveCharacterTextSplitter hence we also get good speed.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, # creates chunk of 500 characters

    chunk_overlap = 100, # overlap 100 chunks from previous created chunk into the next chunk like (Chunk 1 : 1 - 500, Chunk 2 : 401 - 900)

    separators = ["\n\n", "\n", " ", ""], # This is the default separators list we can tweak it also, detailed explanation below.

    keep_separator = True, # By default it is True, It means whether to include the separator like if its splitting at paragraph level ("\n\n") then whether to include this "\n\n" inside the chunk or not. This is not related with the above separators.

    length_function = len, # By default uses "len()" function. It is used to measure the chunk size we defined above, "len()" means characers, so now above "chunk_size = 500" means 500 characters. "chunk_size" depends on what function we are using. We can also use a tokenizer counter to count tokens so above "chunk_size = 500" will no longer means 500 characters, because a token can be a word or a part of word, it depends on the tokenizer itself.

    strip_whitespace = True # Removes leading/trailing whitespace from every chunk

)

# Above we can add more parameters as per needs but these are some most important and mostly used ones


chunks = splitter.split_documents(all_documents) # Converts List[Document] -> List[Document] but now each document page_content is exactly 500 characters and also metadata remains same ,it just copy previous metadata into new one

# These below functions can be used above as per requirements

# split_text() - used when we have single text string. Converts str -> List[str]
# create_documents() - used when we have list of strings. Converts List[str] -> List[Document]

In [9]:
len(chunks) # We can see length is also increased

524

<b><u>Separators in detail</b></u>

separators tells the splitter where it should try to split the text.

1. "\n\n" - means split text paragraph wise if not possible then go to next Separator

2. "\n" - means split the text line by line if not possible then go to next Separator

3. " " - means split at each word or word wise, if can't then go to nect Separator

4. "" - means split character wise, smallest possible chunk that can be created


- These above are default splitters ["\n\n", "\n", " ", ""]. Priority Left -> Right, means it will first try with "\n\n" then the next if the size is exceding the chunk_size we have defined

We can also pass some custom separators if we are working with mardown files, code files, etc.

### 3. Creating Embeddings and Storing in a Vector DB

We have to make sure here that the model we will use for creating embeddings here should be used same while making any query

Here we use HuggingFaceEmbeddings, this is a wrapper which internally uses sentence-transformers and download the model locally and make it ready to convert text to embeddings or vectors

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-small-en-v1.5",
    model_kwargs = {
        "device" : "cpu" # "cuda" for gpu
    },
    encode_kwargs = {
        "normalize_embeddings" : True,
        "batch_size" : 32
    }
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

vectorstore.save_local("faiss_db")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1113.12it/s]


<b><u>Important Parameters for HuggingFaceEmbeddings</b></u>

1. encode_kwargs = {"normalize_embeddings" : True}
    - This is very important paramter it normalize embeddings and make every vector a unit vector means every vector with length or magnitude 1.
    - It means when we talk about a vector its nothing but a list of numbers, and a vector have both direction and magnitude.
    - Now if we have two text like "Python is easy" and "Python is simple" then their vectors will point in the same direction but with difference in their magnitude.
    - So whenever I ask a query like "Python Language" now I want these relevent vectors to get retrieved without caring about their magnitude values

    - How it works mathematically :
        - Suppose a vector [3, 4]
        - Length = √(3)² + (4)² = √25 = 5
        - Divide original values with 5
        - [3/5, 4/5] => [0.6, 0.8]
        - Now its length become 1.

    - This works perfectly with cosine similarity because it measures only the angle between vectors
    - This is useful where only angle or direction matters but if we are using a formula for matching which also differentiates according to magnitude then this may not be good there
    - This also makes computations simpler and this is why models like BGE recomment it

2. encode_kwargs = {"batch_size" : 32}
    - This measures how many chunks are processed in one pass.
    - Like if {"batch_size" : 1} then it work like :
        - Chunk 1 -> Model -> Embedding 1
        - Chunk 2 -> Model -> Emvedding 2
        - ....

    - And if {"batch_size" : 32}
        - Chunk 1-32 -> Model -> 32 Embeddings
        - Chunk 33-65 -> Model -> 32 Embeddings
        - .....

    - It works with both cpu and gpu
    


1. <b><u>What is "from_documents()" ?</b></u>
    - Step 1 - Extract text, we know our chunks are list of Document it will extract the text "page_content" and then make list of it like ["Python is ...", "Agentic AI ..."].

    - Step 2 - Now it generates embeddings for each text present in the list for example [[0.12, -0.45, ...], [0.67, 0.21, ...]].

    - Build FAISS Index. FAISS stores vectors in an efficient data structure and it is optimized for fast similarity search like :

    <pre>
    ID          Vector
    ----------------------------
    0          [0.12,-0.45,...]
    1          [0.67,0.21,...]
    </pre>

    - Step 4 - LangChain also stores the original document object separately like :

    <pre>
    ID          Vector
    ----------------------------
    0           Document(
                    page_content="Python...",
                    metadata={...}
                )

    1           Document(
                    page_content="Machine Learning...",
                    metadata={...}
                )


    </pre>

    - This is crucial because when we make a query like "What is Python" then it will become embedding and then FAISS compares this with all the stored vector, suppose index "0" is the closest then FAISS will return "Vector ID = 0", then LangChain looks up and returns the document object with "Vector ID = 0"

    - This is why retriever returns "Document" object rather than raw vectors

Here we are performing both the steps of creating embeddings and storing it together because LangChain combines "Embedding" and "Storing in Vector DB" both steps together for convenience.

<pre>
"""
FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
"""
</pre>

Here in this code "from_documents()" perform all the steps mentioned above and returns a vectorstore it is not just the FAISS Index but it is a LangChain object that contains :

<pre>
"""
FAISS Object
├── FAISS Index
├── Document Store
├── ID Mapping
└── Embedding Model Reference
"""
</pre>

So it is a complete vector store, not only the vectors which we can save locally with "vectorstore.save_local("faiss_db")" so it will be saved in our Disk and we do not need to rebuild vectors again and again.

Also it stores two files :

<pre>
"""
faiss_db/
├── index.faiss
└── index.pkl
"""
</pre>


1. "index.faiss" - Contains the FAISS index itself (vectors mapped with index)

2. "index.pkl" - Contains everything else :

    - Original Document objects
    - Metadata
    - Mapping between vector IDs and documents


### Extra : Creating Embeddings and Storing in a Cloud based Vector Storage DB

First make sure to create a account on Qdrant cloud and create a free cluster there, after creating the free cluster you will get API Key and Cluster URL save them to env file

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

client = QdrantClient(
    api_key = os.getenv("QDRANT_CLUSTER_API_KEY"),
    url = os.getenv("QDRANT_CLUSTER_URL")
)


cloud_vectorstore = QdrantVectorStore.from_documents(
    documents = chunks,
    embedding = embeddings,
    client = client,
    collection_name = "Agentic_AI_RAG_Course"
)

Now this will create a collection inside the cluster we have created which will store everything, one cluster can contain many collections.

This is done now we will see in the Retrieval Pipeline how we can retrive context from it both locally and cloud based.